In [1]:
# =============================================
# Import packages and libraries
# =============================================
import numpy as np
import pandas as pd
import pyodbc
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
# =============================================
# Read data from SQL Server Database
# =============================================
conn = pyodbc.connect(
    "Driver={SQL Server};"
    "Server=LAPTOP-M34KOLBO\\SQLEXPRESS;"
    "Database=bank_churn;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

query = """
SELECT
    d.Gender, d.Age, d.Salary, l.[Geography] , 
    a.Tenure, a.Balance, a.NumProducts, a.HasCreditCard, a.IsActive,
    d.Churned
FROM demographic d
JOIN account a ON a.CustomerId = d.CustomerId
JOIN [location] l ON l.LocationId = d.LocationId
"""

In [3]:
df = pd.read_sql(query, conn)

C:\Users\Tshedza\AppData\Local\Temp\ipykernel_20520\1168034203.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [4]:
df.head(10)

,Gender,Age,Salary,Geography,Tenure,Balance,NumProducts,HasCreditCard,IsActive,Churned
0,Female,39,90212.38,Spain,9,161993.89,1,False,False,False
1,Male,35,83256.26,Germany,1,119827.49,1,True,True,True
2,Male,47,96517.97,USA,1,64430.06,2,False,True,False
3,Female,30,188258.49,Spain,6,57809.32,1,True,False,False
4,Male,48,74510.65,Spain,10,96048.55,1,True,False,False
5,Male,38,30583.95,USA,9,119827.49,2,False,False,False
6,Male,29,197963.46,USA,3,119827.49,2,True,True,False
7,Female,28,56185.98,Spain,9,119827.49,2,True,True,False
8,Male,39,56214.09,Spain,8,119827.49,2,True,False,False
9,Male,44,154639.72,France,8,119827.49,2,True,True,False


In [5]:
# =============================================
# Split Train and Test Dataset
# =============================================
FRAC = 0.8
SEED = 200

df_train = df.sample(frac = FRAC, random_state = SEED)

df_test = df.drop(df_train.index)
print(len(df_train))
print(len(df_test))

8000
2000


In [6]:
# =============================================
# Feature Engineering in Training
# =============================================
df_train['BalanceSalaryRatio'] = df_train.Balance / df_train.Salary
df_train['TenureByAge'] = df_train.Tenure / df_train.Age

In [7]:
# Display the results
print(df_train[['Balance', 'Salary', 'BalanceSalaryRatio',
                'Tenure', 'Age', 'TenureByAge']].head(10))

        Balance     Salary  BalanceSalaryRatio  Tenure  Age  TenureByAge
8159  133377.80   64050.19            2.082395       0   35     0.000000
6332  119714.25  150135.38            0.797375       3   20     0.150000
8895  132187.73  178331.36            0.741248       1   32     0.031250
5351  119827.49   82393.08            1.454339       7   25     0.280000
4314  119827.49   77108.66            1.554008       9   37     0.243243
3987  111183.53   10063.75           11.047922       5   18     0.277778
3300  119684.88   74532.02            1.605818       6   70     0.085714
370   115076.06   79323.61            1.450716       7   42     0.166667
7647  148363.38  186995.17            0.793408       6   67     0.089552
4763  129431.36   21343.74            6.064137      10   67     0.149254


In [8]:
# =============================================
# Scaling Numerical Features in Training
# =============================================
num_cols = df_train.select_dtypes(include=['int64','float64']).columns
scaler = StandardScaler()
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
# Print results
print(df_train.head(10))

      Gender       Age    Salary Geography    Tenure   Balance  NumProducts  \
8159    Male -0.372496 -0.630673       USA -1.730806  0.574363    -0.907267   
6332    Male -1.795729  0.868108        UK -0.695996  0.003023     0.802541   
8895    Male -0.657142  1.359012     Spain -1.385869  0.524601     0.802541   
5351  Female -1.321318 -0.311315        UK  0.683751  0.007758     0.802541   
4314    Male -0.182731 -0.403319   Germany  1.373624  0.007758     0.802541   
3987    Male -1.985493 -1.570600    Canada -0.006123 -0.353688     0.802541   
3300    Male  2.948381 -0.448180        UK  0.338814  0.001795     0.802541   
370     Male  0.291680 -0.364756        UK  0.683751 -0.190922    -0.907267   
7647    Male  2.663735  1.509853       USA  0.338814  1.200984    -0.907267   
4763    Male  2.663735 -1.374211   Germany  1.718560  0.409343    -0.907267   

      HasCreditCard  IsActive  Churned  BalanceSalaryRatio  TenureByAge  
8159          False      True    False           -0.0345

In [9]:
print("Numerical columns after scaling")
print(df_train[num_cols].head(10))


Numerical columns after scaling
           Age    Salary    Tenure   Balance  NumProducts  BalanceSalaryRatio  \
8159 -0.372496 -0.630673 -1.730806  0.574363    -0.907267           -0.034587   
6332 -1.795729  0.868108 -0.695996  0.003023     0.802541           -0.044991   
8895 -0.657142  1.359012 -1.385869  0.524601     0.802541           -0.045446   
5351 -1.321318 -0.311315  0.683751  0.007758     0.802541           -0.039672   
4314 -0.182731 -0.403319  1.373624  0.007758     0.802541           -0.038865   
3987 -1.985493 -1.570600 -0.006123 -0.353688     0.802541            0.038003   
3300  2.948381 -0.448180  0.338814  0.001795     0.802541           -0.038446   
370   0.291680 -0.364756  0.683751 -0.190922    -0.907267           -0.039702   
7647  2.663735  1.509853  0.338814  1.200984    -0.907267           -0.045024   
4763  2.663735 -1.374211  1.718560  0.409343    -0.907267           -0.002349   

      TenureByAge  
8159    -1.534952  
6332     0.130786  
8895    -1.18792

In [10]:
# =============================================
# Encoding Categorical Features in Training
# =============================================
cat_cols = df_train.select_dtypes(include=['object', 'string', 'bool']).columns.drop(['Churned'])

for col in cat_cols:
    for val in df_train[col].unique():
        df_train[f"{col}_{val}"] = np.where(df_train[col] == val, 1, 0)

df_train = df_train.drop(cat_cols, axis=1)
print(df_train.head(10))

           Age    Salary    Tenure   Balance  NumProducts  Churned  \
8159 -0.372496 -0.630673 -1.730806  0.574363    -0.907267    False   
6332 -1.795729  0.868108 -0.695996  0.003023     0.802541    False   
8895 -0.657142  1.359012 -1.385869  0.524601     0.802541    False   
5351 -1.321318 -0.311315  0.683751  0.007758     0.802541    False   
4314 -0.182731 -0.403319  1.373624  0.007758     0.802541    False   
3987 -1.985493 -1.570600 -0.006123 -0.353688     0.802541    False   
3300  2.948381 -0.448180  0.338814  0.001795     0.802541    False   
370   0.291680 -0.364756  0.683751 -0.190922    -0.907267    False   
7647  2.663735  1.509853  0.338814  1.200984    -0.907267    False   
4763  2.663735 -1.374211  1.718560  0.409343    -0.907267    False   

      BalanceSalaryRatio  TenureByAge  Gender_Male  Gender_Female  \
8159           -0.034587    -1.534952            1              0   
6332           -0.044991     0.130786            1              0   
8895           -0.0454

In [11]:
# =============================================
# Preprocessing Pipeline in Testing
# =============================================
def DfTestPipeline(df_test):
    df_test['BalanceSalaryRatio'] = df_test.Balance / df_test.Salary
    df_test['TenureByAge'] = df_test.Tenure / df_test.Age

    df_test[num_cols] = scaler.transform(df_test[num_cols])

    for col in cat_cols:
        for val in df_test[col].unique():
         df_test[f"{col}_{val}"] = np.where(df_test[col] == val, 1, 0)

    df_test = df_test.drop(cat_cols, axis=1)

    return df_test


df_test = DfTestPipeline(df_test)

# Print results
print(df_test.head(10))


         Age    Salary    Tenure   Balance  NumProducts  Churned  \
2   0.766091 -0.065395 -1.385869 -2.308680     0.802541    False   
3  -0.846907  1.531848  0.338814 -2.585525    -0.907267    False   
5  -0.087849 -1.213335  1.373624  0.007758     0.802541    False   
17  0.481444  0.811623  1.373624  0.993067    -0.907267     True   
18 -1.321318 -0.922164 -0.006123  0.007758     0.802541    False   
27 -0.562260  1.508936 -0.351059 -1.321145    -0.907267     True   
28  0.481444  0.119264  1.373624  0.007758     0.802541    False   
37 -1.036671 -0.944219  1.028687  0.432776     0.802541    False   
40  0.007033 -0.283376 -0.695996  0.760010    -0.907267    False   
46 -0.372496 -1.694913 -1.385869 -0.031266    -0.907267    False   

    BalanceSalaryRatio  TenureByAge  Gender_Male  Gender_Female  \
2            -0.046043    -1.298677            1              0   
3            -0.048961     0.686032            0              1   
5            -0.019725     1.095161            1  

In [12]:
# =============================================
# Re-order columns in Testing and Training
# =============================================
df_train.columns.equals(df_test.columns)
df_test = df_test[df_train.columns]

In [ ]:
# Print results
print("Do columns match:", df_train.columns.equals(df_test.columns))

print("\nTraining data:")
print(df_train.head(10))

print("\nTesting data:")
print(df_test.head(10))

In [ ]:
# =============================================
# Declare Features and Targets
# =============================================
df_train_x = df_train.drop('Churned', axis=1)
df_train_y = df_train['Churned']

df_test_x = df_test.drop('Churned', axis=1)
df_test_y = df_test['Churned']

In [ ]:

# =============================================
# Save Data
# =============================================
artifacts = {
    "X_train": df_train_x,
    "y_train": df_train_y,
    "X_test": df_test_x,
    "y_test": df_test_y
}

joblib.dump(artifacts, 'dataset_bundle.pkl')